<a href="https://colab.research.google.com/github/soberbichler/NLP-Course4Humanities_2025.github.io/blob/main/Analyse_and_Visualize_Datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Analyse and Visualize a Dataset

In [ ]:
import pandas as pd
import string
import nltk
from nltk.corpus import stopwords
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import numpy as np
from wordcloud import WordCloud
import plotly.graph_objects as go

## Importing Data to the Notebook

In order to access our course data, we clone the course GitHub repository to this notebook. Do do so, run the *git clone* cell below:

In [ ]:
!git clone https://github.com/soberbichler/NLP-Course4Humanities_2025.github.io.git

In [ ]:
# @markdown ##### If you copied the path, find the place where you can add it. Run the code and investigate the dataset
import pandas as pd

# Replace 'your_file.xlsx' with the actual path to your Excel file.
df = pd.read_excel('')

# Display the first few rows of the DataFrame to verify it's loaded correctly.
df.head(5)

## Frequenzy Analysis

In [ ]:
# Lower letter
content = df['text'].tolist()
content = [text.lower() for text in content]
print(content[0])

In [ ]:
# Removing punctuations
listofthings = []
for entry in content:
    for c in string.punctuation:
        entry = entry.replace(c, " ")
    words = entry.split()
    listofthings.extend(words)

In [ ]:
print(listofthings)

In [ ]:
# Removing enumerations
words = [word for word in listofthings if not word.isdecimal()]
print(words)

#### Removal of stopwords

In [ ]:

nltk.download('stopwords')
stopger = stopwords.words('german')


newStopwords = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l',
                'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'vgl', '\x97', '•', '■', 'v',
                'beim', 'de','—','ge','la','be','en','que','el','ten','ver','gen','sei','nen','del','nen', 'se','schen','un','land','te','ei','aires',
                'las', 'los', '«']


stopger.extend(newStopwords)

In [ ]:
#print(stopger)

In [ ]:

tokens_without_sw = [word for word in words if word not in stopger]
print(tokens_without_sw)

In [ ]:
# Counting frequencies
counts = Counter(tokens_without_sw)

# Top 20 words
top_20_words = counts.most_common(20)
top_150_words = counts.most_common(150)
print(top_20_words)

## Visualizations

#### 1. Pie chart:

In [ ]:

def create_circle(word_count):
    labels = [word[0] for word in word_count]
    sizes = [word[1] for word in word_count]
    colors = [plt.cm.Spectral(i/float(len(labels))) for i in range(len(labels))]

    plt.figure(figsize=(10, 8))
    plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140)
    plt.axis('equal')
    plt.title('Top 20 most frequent words')
    plt.show()


create_circle(top_20_words)

#### 2. Bar chart:

In [ ]:

def create_bar_chart(word_count):
    labels = [word[0] for word in word_count]
    sizes = [word[1] for word in word_count]

    x = np.arange(len(labels))
    width = 0.75

    fig, ax = plt.subplots(figsize=(12, 8))
    rects = ax.bar(x, sizes, width)

    # Labels hinzufügen, etc.
    ax.set_ylabel('Häufigkeit')
    ax.set_title('Top 20 häufigste Wörter')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha='right')

    # Balken beschriften
    for rect in rects:
        height = rect.get_height()
        ax.annotate('{}'.format(height),
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom')

    return fig, ax
    #plt.show()


fig, ax = create_bar_chart(top_20_words)

In [ ]:
# save
fig.savefig('plot.jpg', format='jpg', dpi=300)
# fig.savefig('plot.png', format='png', dpi=300)

#### 4. Frequency of a word per year

In [ ]:
def tokenize(content):

    wordlist = []
    for word in content.split():
        for char in string.punctuation:
            word = word.strip(char)
        wordlist.append(word)

    words = [word for word in wordlist if not word.isdecimal()]
    tokens_without_sw = [word for word in words if word not in stopger]

    return tokens_without_sw

In [ ]:
df['year'] = df['filename'].str[:4]
df['token'] = df['text'].apply(tokenize)
df

In [ ]:
def counting(tokens):
    counts = Counter(tokens)
    return counts

In [ ]:
df['counts'] = df['token'].apply(counting)
df

In [ ]:
import plotly.express as px

word_to_analyze = 'Amerika'


word_counts = df[['year', 'counts']].copy()
word_counts['count'] = word_counts['counts'].apply(lambda x: x.get(word_to_analyze, 0))
word_counts = word_counts.groupby('year')['count'].sum().reset_index()


fig = px.line(word_counts, x='year', y='count', title=f'Häufigkeit des Wortes "{word_to_analyze}" pro Jahr', height=550)
fig.update_layout(xaxis_title='Jahr', yaxis_title='Häufigkeit')
fig.show()



In [ ]:

entry_counts = df.groupby('year').size().reset_index(name='entry_count')


fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=word_counts['year'], y=word_counts['count'], mode='lines', name=f'Häufigkeit des Wortes "{word_to_analyze}"'))
fig2.add_trace(go.Scatter(x=entry_counts['year'], y=entry_counts['entry_count'], mode='lines', name='Anzahl der Einträge'))

fig2.update_layout(
    title=f'Häufigkeit des Wortes "{word_to_analyze}" und Anzahl der Einträge pro Jahr',
    xaxis_title='Jahr',
    yaxis_title='Werte',
    height=550
)

fig2.show()